# Create and Load DuckDB Tables

In [5]:
import duckdb

DB_PATH = "data/electio.duckdb"


In [6]:
con = duckdb.connect(DB_PATH)

con.execute("""
    CREATE OR REPLACE TABLE dim_commune AS
    SELECT * FROM read_csv_auto('data/gold/election/dim_commune.csv', sep=';')
""")

con.execute("""
    CREATE OR REPLACE TABLE dim_election AS
    SELECT * FROM read_csv_auto('data/gold/election/dim_election.csv', sep=';')
""")

con.execute("""
    CREATE OR REPLACE TABLE dim_candidat AS
    SELECT * FROM read_csv_auto('data/gold/election/dim_candidat.csv', sep=';')
""")

con.execute("""
    CREATE OR REPLACE TABLE dim_nuance AS
    SELECT * FROM read_csv_auto('data/gold/election/dim_nuance.csv', sep=';')
""")

con.execute("""
    CREATE OR REPLACE TABLE fact_participation AS
    SELECT * FROM read_csv_auto('data/gold/election/fact_participation.csv', sep=';')
""")

con.execute("""
    CREATE OR REPLACE TABLE fact_resultats_candidat AS
    SELECT * FROM read_csv_auto('data/gold/election/fact_resultats_candidat.csv', sep=';')
""")

con.execute("""
    CREATE OR REPLACE TABLE dim_indicateur_securite AS
    SELECT * FROM read_csv_auto('data/gold/security/dim_indicateur_securite.csv', sep=';')
""")

con.execute("""
    CREATE OR REPLACE TABLE fact_securite AS
    SELECT * FROM read_csv_auto('data/gold/security/fact_securite.csv', sep=';')
""")

con.execute("""
    CREATE OR REPLACE TABLE fact_demographie AS
    SELECT * FROM read_csv_auto('data/gold/security/fact_demographie.csv', sep=';')
""")

print(con.execute("SHOW TABLES").fetchdf())

con.close()


                      name
0             dim_candidat
1              dim_commune
2             dim_election
3  dim_indicateur_securite
4               dim_nuance
5         fact_demographie
6       fact_participation
7  fact_resultats_candidat
8            fact_securite


In [7]:
con = duckdb.connect(DB_PATH)

for table in [
    "dim_commune",
    "dim_election",
    "dim_candidat",
    "dim_nuance",
    "fact_participation",
    "fact_resultats_candidat",
    "dim_indicateur_securite",
    "fact_securite",
    "fact_demographie",
]:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

con.close()


dim_commune: 305 rows
dim_election: 6 rows
dim_candidat: 20 rows
dim_nuance: 89 rows
fact_participation: 1680 rows
fact_resultats_candidat: 10894 rows
dim_indicateur_securite: 18 rows
fact_securite: 37125 rows
fact_demographie: 2475 rows


In [8]:
con = duckdb.connect(DB_PATH)

result = con.execute("""
    SELECT c.nom, c.prenom, SUM(f.voix) AS total_voix
    FROM fact_resultats_candidat f
    JOIN dim_candidat c
      ON f.id_candidat = c.id_candidat
    JOIN dim_election e
      ON f.id_election = e.id_election
    WHERE e.annee_election = 2022
      AND e.tour = 1
    GROUP BY c.nom, c.prenom
    ORDER BY total_voix DESC
""").fetchdf()

print(result)

con.close()


              nom    prenom  total_voix
0          MACRON  Emmanuel    278243.0
1       MÉLENCHON  Jean-Luc    229036.0
2          LE PEN    Marine    150463.0
3         ZEMMOUR      Éric     74168.0
4           JADOT   Yannick     51907.0
5        PÉCRESSE   Valérie     50263.0
6        LASSALLE      Jean     17572.0
7   DUPONT-AIGNAN   Nicolas     16487.0
8         ROUSSEL    Fabien     15938.0
9         HIDALGO      Anne     15895.0
10         POUTOU  Philippe      5267.0
11        ARTHAUD  Nathalie      3774.0
